In [ ]:
import os
os.chdir('..')
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box, Polygon, Point
from pyproj import Transformer, CRS
import math

In [6]:
# 0. 定义研究区 (Study Area)

tb_area = gpd.read_file('data/hma-extent/HMA/hma_gtng_202307_subregions.gpkg')
minx, miny, maxx, maxy = tb_area.total_bounds
print(f"Study Area Bounds: lon[{minx:.2f}, {maxx:.2f}], lat[{miny:.2f}, {maxy:.2f}]")

buffer_m = 10000       
dx_utm = 100000        
dy_utm = 100000      

Study Area Bounds: lon[67.00, 104.80], lat[26.76, 46.00]


In [7]:
# 1. 生成基于 UTM 等距投影的网格 (100km x 100km)

def get_utm_epsg(lon, lat):
    zone_number = math.floor((lon + 180) / 6) + 1
    epsg_code = 32600 + zone_number if lat >= 0 else 32700 + zone_number
    return f"EPSG:{epsg_code}"

def generate_grid_utm(lonmin, latmin, lonmax, latmax, dx, dy):
    tiles = []
    current_lat = latmin
    
    while current_lat < latmax:
        current_lon = lonmin
        row_max_lat = current_lat
        
        while current_lon < lonmax:
            epsg = get_utm_epsg(current_lon, current_lat)
            
            proj_wgs84 = CRS("EPSG:4326")
            proj_utm = CRS(epsg)
            transformer_to_utm = Transformer.from_crs(proj_wgs84, proj_utm, always_xy=True)
            transformer_to_wgs84 = Transformer.from_crs(proj_utm, proj_wgs84, always_xy=True)
            
            utm_x, utm_y = transformer_to_utm.transform(current_lon, current_lat)
            
            utm_x2 = utm_x + dx
            utm_y2 = utm_y + dy
            
            poly_utm = box(utm_x, utm_y, utm_x2, utm_y2)
            
            poly_utm_buffered = poly_utm.buffer(2000)
            
            x_utm_coords, y_utm_coords = poly_utm_buffered.exterior.xy
            lon_coords, lat_coords = transformer_to_wgs84.transform(x_utm_coords, y_utm_coords)
            poly_wgs84 = Polygon(zip(lon_coords, lat_coords))
            
            tiles.append({
                'geometry': poly_wgs84,
                'proj': epsg
            })
            
            next_lon, next_lat = transformer_to_wgs84.transform(utm_x2, utm_y2)
            current_lon = next_lon
            
            if next_lat > row_max_lat:
                row_max_lat = next_lat
                
        current_lat = row_max_lat
        
    return gpd.GeoDataFrame(tiles, crs="EPSG:4326")


In [8]:
# 2. 生成基于 WGS84 的经纬度网格 (1° x 1°)

def generate_grid_wgs84(xmin, ymin, xmax, ymax, dx=1, dy=1):
    tiles = []
    x_vals = np.arange(xmin, xmax, dx)
    y_vals = np.arange(ymin, ymax, dy)
    
    for x in x_vals:
        for y in y_vals:
            tiles.append(box(x, y, x + dx, y + dy))
            
    return gpd.GeoDataFrame(geometry=tiles, crs="EPSG:4326")

print("Generating grids(WGS84)")
tb_tiles = generate_grid_utm(minx, miny, maxx, maxy, dx_utm, dy_utm)
tb_tiles_wgs84 = generate_grid_wgs84(minx, miny, maxx, maxy, 1, 1)


Generating grids(WGS84)


In [9]:
# 3. 移除空网格并计算面积

def filter_and_calc_area(gdf_tiles, study_area_gdf):
    intersected = gpd.sjoin(gdf_tiles, study_area_gdf, how="inner", predicate="intersects")
    intersected = intersected[gdf_tiles.columns].drop_duplicates()
    
    intersected['area_km2'] = intersected.geometry.to_crs("EPSG:6933").area / 1e6
    
    intersected = intersected.reset_index(drop=True)
    intersected['tile_id'] = [str(i + 1).zfill(3) for i in range(len(intersected))]
    
    return intersected

print("Filtering intersections...")
tb_tiles = filter_and_calc_area(tb_tiles, tb_area)
tb_tiles_wgs84 = filter_and_calc_area(tb_tiles_wgs84, tb_area)


Filtering intersections...


In [10]:
# 4. 对 Tiles 增加外围 Buffer

def add_buffer_to_tiles(gdf_tiles, dist_m):
    buffered_geometries = []
    
    for idx, row in gdf_tiles.iterrows():
        epsg = row.get('proj', 'EPSG:3857') 
        single_gdf = gpd.GeoDataFrame(geometry=[row.geometry], crs="EPSG:4326")
        single_utm = single_gdf.to_crs(epsg)
        buffered_geom = single_utm.geometry.buffer(dist_m).envelope
        buffered_wgs84 = gpd.GeoDataFrame(geometry=buffered_geom, crs=epsg).to_crs("EPSG:4326")
        buffered_geometries.append(buffered_wgs84.geometry.iloc[0])
        
    gdf_buf = gdf_tiles.copy()
    gdf_buf['geometry'] = buffered_geometries

    gdf_buf['area_km2'] = gdf_buf.geometry.to_crs("EPSG:6933").area / 1e6
    
    return gdf_buf

print("Applying buffers...")
tb_tiles_buf = add_buffer_to_tiles(tb_tiles, buffer_m)



Applying buffers...


In [ ]:
# 5. 导出 Shapefiles

out_dir = "data"
os.makedirs(out_dir, exist_ok=True)

path_buf = os.path.join(out_dir, 'hma_tiles_buf.gpkg')
tb_tiles_buf.to_file(path_buf, driver='GPKG', layer='hma_tiles_buf')
print(f"Exported: {os.path.abspath(path_buf)}")

# 导出 WGS84 瓦片
# path_wgs84 = os.path.join(out_dir, 'tibet_tiles_wgs84.gpkg')
# tb_tiles_wgs84.to_file(path_wgs84, driver='GPKG', layer='tibet_tiles_wgs84')
# print(f"Exported: {path_wgs84}")

print("All tasks finished successfully!")

Exported: d:\A_WaterCode\HMA-GIS-Data\data\hma_tiles_buf.gpkg
All tasks finished successfully!
